Load or Prepare Your Data

In [0]:
# Import Spark and Pandas
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

# Load curated dataset from STEDI database
df_spark = spark.table("stedi.curated_step_data")

# Show schema to verify
df_spark.printSchema()
df_spark.show(5)



Load or Prepare Your Data

Hyperparameter Tuning for Random Forest

In [0]:
# Convert Spark DataFrame to Pandas, excluding problematic timestamp column
# Only select columns needed for features and labels
selected_cols = ['distance_cm', 'step_label']
df = df_spark.select(selected_cols).toPandas()

# Features
X = df[['distance_cm']]  # replace with ['x','y','z'] if accelerometer columns exist

# Labels
y = df['step_label']

# Ensure numeric 2D array for scikit-learn
X = np.array(X, dtype=float)
y = np.array(y, dtype=int)

print(X.shape, y.shape)


Compare Tuned Models

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)



Select and Save the Best Model

In [0]:
# Drop rows with NaN in X_train or y_train before fitting
from sklearn.utils import resample

# Find non-NaN indices
valid_idx = ~np.isnan(X_train).ravel() & ~np.isnan(y_train)
X_train_clean = X_train[valid_idx]
y_train_clean = y_train[valid_idx]

log_reg_grid.fit(X_train_clean, y_train_clean)


Model Evaluation & Ethics Reflection

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

log_reg_params = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"]
}

log_reg_grid = GridSearchCV(
    LogisticRegression(max_iter=300),
    log_reg_params,
    cv=3,
    scoring="accuracy"
)

# Use cleaned training data without NaNs
log_reg_grid.fit(X_train_clean, y_train_clean)

log_reg_best_params = log_reg_grid.best_params_
log_reg_best_score = log_reg_grid.best_score_

print("Logistic Regression Best Params:", log_reg_best_params)
print("Logistic Regression Best CV Score:", log_reg_best_score)


In [0]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(),
    rf_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

rf_best_params = rf_grid.best_params_
rf_best_score = rf_grid.best_score_

print("Random Forest Best Params:", rf_best_params)
print("Random Forest Best CV Score:", rf_best_score)


In [0]:
results = {
    "Logistic Regression (tuned)": log_reg_best_score,
    "Random Forest (tuned)": rf_best_score
}

print(results)


In [0]:
import joblib

if rf_best_score > log_reg_best_score:
    best_model = rf_grid.best_estimator_
    best_model_name = "Random Forest"
else:
    best_model = log_reg_grid.best_estimator_
    best_model_name = "Logistic Regression"

print("Best Model:", best_model_name)

# Save model
joblib.dump(best_model, "stedi_best_model.pkl")


In [0]:
from sklearn.metrics import accuracy_score, classification_report

# Predict on test set
y_pred = best_model.predict(X_test)

# Accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", test_accuracy)

# Detailed report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Model Evaluation Report 

After tuning both models using GridSearchCV, Random Forest performed slightly better than Logistic Regression. The best cross-validation accuracy for Logistic Regression was approximately 0.9510, while Random Forest achieved approximately 0.9513. Although the difference is small, Random Forest had the highest validation score and was selected as the final model.

The best Random Forest parameters included 50 trees with no maximum depth restriction and default splitting rules. These settings allowed the model to capture nonlinear relationships in the distance sensor data. When evaluated on the test set, the model maintained similar performance, confirming that the tuning process generalized well and did not overfit the training data.

If more time were available, I would test additional hyperparameters, try Gradient Boosting models, or include additional features beyond distance_cm to improve performance further.

Ethics and Fairness Reflection

Hyperparameter tuning can unintentionally introduce bias if we optimize only for overall accuracy while ignoring how the model performs on different groups or conditions. For example, if certain sensor patterns are underrepresented in the data, the model may perform worse for those cases without us noticing. Transparency in documenting parameter choices and validation results helps ensure accountability and reproducibility.

Careful evaluation aligns with gospel principles of integrity and honesty. Just as discipleship requires consistent effort and truthful self-reflection, model development requires careful testing and responsible reporting of results. Ethical machine learning means measuring performance clearly and avoiding shortcuts that could mislead others.